In [1]:
import polars as pl

def print_era_safety_tables_final(parquet_path: str, top_n: int = 10):
    lf = pl.scan_parquet(parquet_path)
    target_eras = [20240301, 20240901, 20250301]
    
    display_cols = [
        "side", "ma_int", "entry_lookback_units", "exit_window_h", 
        "SL", "TP", "total_pos", "balance", "max_drawdown"
    ]

    for era in target_eras:
        print(f"\n" + "="*80)
        print(f"  ERA: {era} | RANKED BY LOWEST DRAWDOWN")
        print("="*80)

        # Filtering for statistically significant samples
        era_table = (
            lf.filter(
                (pl.col("era_int") == era) & 
                (pl.col("total_pos") > 15) &
                (pl.col("total_pos") < 1000)
            )
            .sort("max_drawdown", descending=False)
            .head(top_n)
            .collect()
        )

        if era_table.is_empty():
            print(f"No valid configurations found for Era {era}.")
        else:
            # Display the table
            print(era_table.select(display_cols))
            
            # THE FIX: Access the first element to get the raw float value
            best_dd_series = era_table["max_drawdown"]
            if best_dd_series.is_empty():
                print(f"\n>>> Era {era} Alpha: No drawdown data available")
            else:
                best_dd_value = best_dd_series[0]
                print(f"\n>>> Era {era} Alpha: Best DD achieved was {best_dd_value:.2%}")

# Usage
results_path = r'C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260321_131807_01\master_metrics.parquet'
print_era_safety_tables_final(results_path)


  ERA: 20240301 | RANKED BY LOWEST DRAWDOWN
shape: (10, 9)
┌──────┬────────┬────────────────┬───────────────┬───┬─────┬───────────┬────────────┬──────────────┐
│ side ┆ ma_int ┆ entry_lookback ┆ exit_window_h ┆ … ┆ TP  ┆ total_pos ┆ balance    ┆ max_drawdown │
│ ---  ┆ ---    ┆ _units         ┆ ---           ┆   ┆ --- ┆ ---       ┆ ---        ┆ ---          │
│ i8   ┆ i32    ┆ ---            ┆ i32           ┆   ┆ f32 ┆ i32       ┆ f32        ┆ f32          │
│      ┆        ┆ i32            ┆               ┆   ┆     ┆           ┆            ┆              │
╞══════╪════════╪════════════════╪═══════════════╪═══╪═════╪═══════════╪════════════╪══════════════╡
│ -1   ┆ 14     ┆ 8              ┆ 24            ┆ … ┆ 0.6 ┆ 81        ┆ 147.568909 ┆ 0.053645     │
│ -1   ┆ 14     ┆ 8              ┆ 24            ┆ … ┆ 0.6 ┆ 81        ┆ 147.568909 ┆ 0.053645     │
│ -1   ┆ 14     ┆ 8              ┆ 12            ┆ … ┆ 0.6 ┆ 81        ┆ 147.568909 ┆ 0.053645     │
│ -1   ┆ 12     ┆ 8            

In [3]:
import polars as pl

def print_era_safety_tables_stoch(parquet_path: str, top_n: int = 15):
    lf = pl.scan_parquet(parquet_path)
    target_eras = [20240301, 20240901, 20250301]
    
    display_cols = [
        "side", "ma_int", "entry_lookback_units", "exit_window_h", 
        "SL", "TP", "total_pos", "balance", "max_drawdown"
    ]

    for era in target_eras:
        print(f"\n" + "="*80)
        print(f"  ERA: {era} | STOCHASTIC K6 ONLY | RANKED BY LOWEST DD")
        print("="*80)

        era_table = (
            lf.filter(
                (pl.col("era_int") == era) & 
                (pl.col("stoch_key").str.contains("k6_")) &           # Isolate K6 Logic
                (pl.col("total_pos") > 15) & 
                (pl.col("total_pos") < 1000)
            )
            .sort("max_drawdown", descending=False)
            .head(top_n)
            .collect()
        )

        if era_table.is_empty():
            print(f"No Stoch K6 configurations found for Era {era}.")
        else:
            print(era_table.select(display_cols))
            
            # THE DEFINITIVE FIX:
            # We index the Series with to get the float before formatting.
            best_dd_series = era_table["max_drawdown"]
            if best_dd_series.is_empty():
                print(f"\n>>> Era {era} Alpha: No drawdown data available")
            else:
                best_dd_value = best_dd_series[0]
                print(f"\n>>> Era {era} Alpha: Best DD achieved was {best_dd_value:.2%}")

# Usage
results_path = r'C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260321_131807_01\master_metrics.parquet'
print_era_safety_tables_stoch(results_path)


  ERA: 20240301 | STOCHASTIC K6 ONLY | RANKED BY LOWEST DD
shape: (15, 9)
┌──────┬────────┬────────────────┬───────────────┬───┬─────┬───────────┬────────────┬──────────────┐
│ side ┆ ma_int ┆ entry_lookback ┆ exit_window_h ┆ … ┆ TP  ┆ total_pos ┆ balance    ┆ max_drawdown │
│ ---  ┆ ---    ┆ _units         ┆ ---           ┆   ┆ --- ┆ ---       ┆ ---        ┆ ---          │
│ i8   ┆ i32    ┆ ---            ┆ i32           ┆   ┆ f32 ┆ i32       ┆ f32        ┆ f32          │
│      ┆        ┆ i32            ┆               ┆   ┆     ┆           ┆            ┆              │
╞══════╪════════╪════════════════╪═══════════════╪═══╪═════╪═══════════╪════════════╪══════════════╡
│ -1   ┆ 14     ┆ 8              ┆ 24            ┆ … ┆ 0.6 ┆ 81        ┆ 147.568909 ┆ 0.053645     │
│ -1   ┆ 15     ┆ 8              ┆ 72            ┆ … ┆ 0.6 ┆ 81        ┆ 147.568909 ┆ 0.053645     │
│ -1   ┆ 13     ┆ 8              ┆ 72            ┆ … ┆ 0.6 ┆ 81        ┆ 147.568909 ┆ 0.053645     │
│ -1   ┆ 15     